# Droplet rebound simulation (SLA experiments) - Run

This notebook is used to run the actual simulations for the droplet rebound test case. The simulation will be restarted from a previously computed initial state, see `DropletReboundInit.ipynb`

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,"[ [OMP_PROC_BIND, spread] ]",\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletRebound", myBatch);

Project name is set to 'DropletRebound'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\DropletRebound'.


In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setting up restart

Choose if this notebook should do a restart. 

In [ ]:
bool restartRun = true;

In [ ]:
//OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound");
//OpenOrCreateDatabase(@"\\dc1\scratch\Smuda\exchange_db");

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

#0: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart6*	05/08/2024 19:31:34	c36ced3a...
#1: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart5*	04/19/2024 14:57:43	42640954...
#2: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4*	04/18/2024 11:39:26	cb667911...
#3: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart3*	04/17/2024 11:27:29	d1016ab7...
#4: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart2*	04/16/2024 11:43:08	005048e6...
#5: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_c4t4_restart3*	04/06/2024 19:28:58	a73dc751...
#6: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_c4t4_restart2*	04/05/2024 13:19:42	fd4010c4...
#7: DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI10_c4t4_restart_withReInit1*	04/04/2024 11:00:

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.Pick(0).Delete(true)
//sessions.Pick(2).Copy(databases.Pick(2))

In [ ]:
//var restartSession = sessions.Where(sess => sess.Name.Contains("dropVelocity100vH")).Single();
var restartSession = databases.Pick(0).Sessions.Pick(0);
restartSession

DropletRebound	DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart5*	04/19/2024 14:57:43	42640954...

In [ ]:
// List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
// restartSessionList.Add(restartSession);
// Guid restartID = restartSession.RestartedFrom;
// while(restartID != Guid.Empty) {
//     var rSess = sessions.Where(sess => sess.ID == restartID).Single();
//     restartSessionList.Add(rSess);
//     restartID = rSess.RestartedFrom;
// }
// restartSessionList

In [ ]:
//restartSession.GetSessionDirectory()
//restartSession.Copy(databases.Pick(1));

In [ ]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(2).Do()

automated naming of restart sessions

In [ ]:
Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
procSIs.Push(restartSession);
var currSI = restartSession;
var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
while(!rSIs.IsNullOrEmpty()) {
    var rSI = rSIs.Single();
    procSIs.Push(rSI);
    currSI = rSI;
    rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
}
int restartNum = procSIs.Count;

string orgName = restartSession.Name;
string restartName;
if (restartNum == 1) {
    //restartName = orgName.Substring(0, orgName.Length - 10); // remoinvg _InitState
//} else if (restartNum == 1){
    restartName = orgName.Substring(0, orgName.Length) + "_restart"; // + restartNum; 
} else {
    restartName = orgName.Substring(0, orgName.Length - 1) + restartNum; 
}

restartName

DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart6

In [ ]:
//restartName = "DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4_checkUpdateFix"

## Boundary Conditions

0: pressure outlet, 1: velocity inlet

In [ ]:
static int BC_tag2_top = 0;
static int BC_tag4_front = 0;

## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [ ]:
static public Grid3D RotatingDiskSector_CartesianCutOut(double radiusOP, double l_radial, double l_azimuthal, double h_axial, int res_radial, int res_azimuthal, int res_axial) {

    double[] xNodes = GenericBlas.Linspace(radiusOP - (l_radial / 2.0), radiusOP + (l_radial / 2.0), res_radial + 1);    // radial direction
    double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 2.0, l_azimuthal / 2.0, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSector3D_CartesianCutOut_{res_radial}x{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");

    if (BC_tag2_top == 0)
        grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    if (BC_tag2_top == 1)
        grd.EdgeTagNames.Add(2, "velocity_inlet_top");

    grd.EdgeTagNames.Add(3, "velocity_inlet_back");

    if (BC_tag4_front == 0)
        grd.EdgeTagNames.Add(4, "pressure_outlet_front");
    if (BC_tag4_front == 1)
        grd.EdgeTagNames.Add(4, "velocity_inlet_front");
   
    grd.EdgeTagNames.Add(5, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(6, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.z.Abs() <= 1e-8)
            et = 1;
        if ((X.z - h_axial).Abs() <= 1e-8)
            et = 2;
        if (((X.x - radiusOP) + (l_radial / 2.0)).Abs() <= 1e-8)
            et = 3;
        if (((X.x - radiusOP) - (l_radial / 2.0)).Abs() <= 1e-8)
            et = 4;
        if ((X.y + (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 5;
        if ((X.y - (l_azimuthal / 2.0)).Abs() <= 1e-8)
            et = 6;
        // if ((X.y + (l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 5;
        // if ((X.y - (2.0 * l_azimuthal / 3.0)).Abs() <= 1e-8)
        //     et = 6;

        return et;
    });    

    return grd;
}

## Simulation setup

In [ ]:
double radiusOP = 78e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.19;  
double kinViscosity = 1.52e-5; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double omega = 1820.5128205128206; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

9.137448098155134E-05

Reynolds number at the point of injection

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

853.6300196960717

Boundary layer (BL) thickness

In [ ]:
double zBL = 5.5;
double zBLstar = zBL * Lstar;
zBLstar

0.0005025596453985323

Height of the computational domain (roughly 3 times the BL thickness)

In [ ]:
double zTop = 15;
double zTopStar = zTop * Lstar;
zTopStar

0.00137061721472327

## Prescribed boundary conditions for the flow field 

The B.C. are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray HAMcoeff_velU = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffU.txt");
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");
MultidimensionalArray HAMcoeff_velW = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffW.txt");
MultidimensionalArray HAMcoeff_P = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffP.txt");

In [ ]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX();
vonKarmanHAM_velX.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity, density);
var vonKarmanHAM_velY = new RotatingDiskBoundaryLayerConditionsHAM_VelocityY();
vonKarmanHAM_velY.SetData(HAMcoeff_velU, HAMcoeff_velV, omega, kinViscosity, density);
var vonKarmanHAM_velZ = new RotatingDiskBoundaryLayerConditionsHAM_VelocityZ();
vonKarmanHAM_velZ.SetData(HAMcoeff_velW, omega, kinViscosity, density);
var vonKarmanHAM_P = new RotatingDiskBoundaryLayerConditionsHAM_PressureP();
vonKarmanHAM_P.SetData(HAMcoeff_P, omega, kinViscosity, density);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [ ]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 1820.5128205128206; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velX = - rOnRotDisk * Math.Sin(theta) * omega;" +
    "return velX; } "
);

In [ ]:
Formula RotatingDiskVelocityY = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { " + 
    "double omega = 1820.5128205128206; "  + 
    "double theta = Math.Atan2(X[1], X[0]); "  + 
    "double rOnRotDisk = Math.Sqrt(X[0].Pow2() + X[1].Pow2()); " + 
    "double velY = rOnRotDisk * Math.Cos(theta) * omega;" +
    "return velY; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [ ]:
double velW_top = -0.88447342;
double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
velWstar_top

-0.14713075072584397

In [ ]:
Formula VelocityZ_top = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -0.14713075072584397; } "
);

## Initial conditions for the injected droplet 

If not restarted from precomputed initial state with impact velocity for the droplet.

In [ ]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 78e-3;" +
    "double radiusDrop = 0.1872e-3;" +   // changed from 0.205e-3
    "double initHeight = 60e-5;" +  // changed from 75e-5
    "return Math.Sqrt((X[0] - radiusOP).Pow2() + X[1].Pow2() + (X[2] - initHeight).Pow2()) - radiusDrop; } "
);

In [ ]:
Formula InitVelocity = new Formula(
    "VelZ",
    false,
    "double VelZ(double[] X) { return -1.12; } "
);

In [ ]:
Formula GravityValue = new Formula(
    "GravZ",
    false,
    "double GravZ(double[] X) { return -9.81; } "
);

## Control settings

In [ ]:
double densityDrop = 998.37;
double sigma = 72.9e-3;

int res_global = 8;
int AMRlevel_disk = 1;
int AMRlevel_drop = 1;
double hmin = (zTopStar / res_global) / (AMRlevel_drop + 1);

int k = 3;

double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(densityDrop, density, sigma, hmin, k+1);
dt_sigma

3.312760419594806E-06

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();


C.SetDGdegree(k);
C.FieldOptions.Add(VariableNames.GravityZ, new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});


// physical parameters
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = densityDrop * 0.000001028;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.IncludeConvection = true;

    
// set grid
double l_radial = zTopStar; 
double l_azimuthal = zTopStar;
// double l_azimuthal = (3.0/2.0) * zTopStar;
double h_axial = zTopStar; 

int res_radial = 1 * res_global; 
int res_azimuthal = 1 * res_global;
// int res_azimuthal = (3 * res_global) / 2;
int res_axial = 1 * res_global;


if (restartRun) {
    C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, 350);

} else {
    // set grid
    Grid3D grd = RotatingDiskSector_CartesianCutOut(radiusOP, l_radial, l_azimuthal, h_axial, res_radial, res_azimuthal, res_axial);
    C.SetGrid(grd);

    // initial conditions
    C.AddInitialValue("VelocityX#B", vonKarmanHAM_velX);
    C.AddInitialValue("VelocityY#B", vonKarmanHAM_velY);
    C.AddInitialValue("VelocityZ#B", vonKarmanHAM_velZ);
    // C.AddInitialValue("Pressure#B", vonKarman_P);

    C.AddInitialValue("Phi", PhiFunc);
    C.AddInitialValue("VelocityZ#A", InitVelocity);

}

C.AddInitialValue("GravityZ#A", GravityValue);
C.AddInitialValue("GravityZ#B", GravityValue);

// boundary conditions
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityY#B", RotatingDiskVelocityY);

if (BC_tag2_top == 1) {
    C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_top", "VelocityZ#B", vonKarmanHAM_velZ);
}

if (BC_tag4_front == 1) {
    C.AddBoundaryValue("velocity_inlet_front", "VelocityX#B", vonKarmanHAM_velX);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityY#B", vonKarmanHAM_velY);
    C.AddBoundaryValue("velocity_inlet_front", "VelocityZ#B", vonKarmanHAM_velZ);
}

C.AddBoundaryValue("velocity_inlet_back", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_back", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_back", "VelocityZ#B", vonKarmanHAM_velZ);

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#B", vonKarmanHAM_velX);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityY#B", vonKarmanHAM_velY);
C.AddBoundaryValue("velocity_inlet_upstream", "VelocityZ#B", vonKarmanHAM_velZ);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityY#B", vonKarman_velY);
// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityZ#B", vonKarman_velZ);
// C.AddBoundaryValue("pressure_dirichlet_downstream", "Pressure#B", vonKarman_P);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.ReInitPeriod = 1;

// C.SkipSolveAndEvaluateResidual = true;

// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.Timestepper_LevelSetHandling = LevelSetHandling.None;
// C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 5e-7;
C.NoOfTimesteps = 1000;
    
{
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_disk });
    C.AMR_startUpSweeps = AMRlevel_drop;
}

// C.PostprocessingModules.Add(new EnergyLogging());
//C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers";
    
if (restartRun) 
    C.SessionName = restartName;
else {
    C.SessionName = $"DropletRebound_8x8x8AMR{AMRlevel_drop}_k{k}_dropVelocity100vH_NeKs_ReI{C.ReInitPeriod}_2dt";
    // C.SessionName = $"RotatingBL_8x8x8AMR{AMRlevel}_k{k}_withoutGravity_NeKs_c4t4";
    // C.SessionName = $"DropletFreeFall_8x8x8AMR{AMRlevel}_k{k}_dropVelocity100vH_NeKs_withReInit{C.ReInitPeriod}_c4t1";
}
    
    
Controls.Add(C);

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart6


In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 4;

    int numThreads = 4;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch); 
}

Deployments so far (0): ;
Success: 0
job submit count: 0
unable to determine job status - unknown
Deploying job DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart6 ... 
Opening existing database 'D:\local\DropletRebound'.
Opening existing database '\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\DropletRebound'.
Set Database: { Session Count = 25; Grid Count = 72; Path = \\dc3\userspace\smuda\hpccluster\DropletRebound }
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletRebound-XNSE_Solver2024May08_155139.119174
copied 42 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.

